In [ ]:
%matplotlib inline
import os
import numpy as np
from matplotlib import pyplot as plt
import camb
from camb import model, initialpower
from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline

In [ ]:
rs=[3, 1, 0] # redshift values for which non-linear power spectrum to be calculated
lst=["-","--","dotted"] # linestyles corresponding to each redshift value, below

In [ ]:
pars = camb.set_params(H0=67.8, ombh2=0.0482, omch2=0.308, ns=0.965) # parameters from DM-L50-N128 param.txt file

pars.set_matter_power(redshifts=rs, kmax=2.0) # these lines follow the descriptions on https://camb.readthedocs.io/en/latest/CAMBdemo.html
results = camb.get_results(pars)
pars.NonLinear = model.NonLinear_both
results.calc_power_spectra(pars)
kh_nonlin, z_nonlin, pk_nonlin = results.get_matter_power_spectrum(minkh=1e-4, maxkh=4, npoints=200)

In [ ]:
plt.figure(figsize=(6,4))
for i, (redshift, line) in enumerate(zip(z_nonlin, lst)):
    plt.loglog(kh_nonlin, pk_nonlin[i, :], color='tab:blue', ls=line, label="z: " + str(z_nonlin[i]))
plt.xlabel('k/h Mpc')
plt.legend()
plt.grid(alpha=0.6,linestyle="--")
plt.ylabel('Matter power spectrum')
plt.savefig('powspec.pdf', dpi=100, bbox_inches='tight')

In [ ]:
def readcomp(file):
    dat=np.loadtxt(file)
    return dat[:,0]+1j*dat[:,1]

In [ ]:
powspecC=np.loadtxt("powspec3d8C.txt")
K_centresC = powspecC[:,0]
Pk_binnedC = powspecC[:,1]
powspecG=np.loadtxt("powspec3d8G.txt")
K_centresG = powspecG[:,0]
Pk_binnedG = powspecG[:,1]

In [ ]:
pars = camb.set_params(H0=67.8, ombh2=0.0482, omch2=0.308, ns=0.965) # parameters from DM-L50-N128 param.txt file

pars.set_matter_power(redshifts=rs, kmax=2.0) # these lines follow the descriptions on https://camb.readthedocs.io/en/latest/CAMBdemo.html
results = camb.get_results(pars)
pars.NonLinear = model.NonLinear_both
results.calc_power_spectra(pars)
kh_nonlin, z_nonlin, pk_nonlin = results.get_matter_power_spectrum(minkh=0.16, maxkh=4.03, npoints=200)

In [ ]:
cs=CubicSpline(kh_nonlin,pk_nonlin[0, :])
inter=cs(K_centresC)

In [ ]:
def prop(x,a):
    return x*a

In [ ]:
poptC,pcovC=curve_fit(prop,Pk_binnedC,inter,p0=[3])
poptG,pcovG=curve_fit(prop,Pk_binnedG,inter,p0=[3])

In [ ]:
plt.figure(figsize=(6,4))
plt.loglog(K_centresC, Pk_binnedC*poptC[0],label="DM-L50-N128, CIC",color="tab:red",linestyle="--")
plt.loglog(K_centresG, Pk_binnedG*poptG[0],label="DM-L50-N128, GSC",color="tab:green",linestyle="--")
for i, (redshift, line) in enumerate(zip(z_nonlin, lst)):
    if (z_nonlin[i]==0):
        plt.loglog(kh_nonlin, pk_nonlin[i, :], color='tab:blue', ls=line, label="CAMB, z=0")
plt.text(0.25,250,"$\\sigma_{\\text{CIC}}=$"+str(round(np.sqrt(np.diag(pcovC))[0],4)),fontsize=13,color="tab:red")
plt.text(0.2435,160,"$\\sigma_{\\text{GSC}}=$"+str(round(np.sqrt(np.diag(pcovG))[0],4)),fontsize=13,color="tab:green")
plt.legend()
plt.xlabel('k/h Mpc')
plt.legend()
plt.grid(alpha=0.6,linestyle="--")
plt.ylabel('Matter power spectrum at z=0')
plt.savefig('final.pdf', dpi=150, bbox_inches='tight')

In [ ]:
def _fourier_grid_box(nvox, boxsize, hermitian = False):
    r'''
    Construct a Fourier grid consistent with a real-space mesh that spans
    ``boxsize`` with ``nvox`` cells, allowing anisotropic cell sizes.

    Parameters
    ----------
    nvox : ndarray of shape (3,)
        Number of grid cells in each dimension.
    boxsize : float or ndarray of shape (3,)
        Physical size of the periodic box [Mpc/h].
    hermitian : bool, optional
        If ``True``, use the reduced ``rfftn`` frequencies along the last
        axis.

    Returns
    -------
    tuple of ndarray
        Fourier wavevector components ``kvec`` and their magnitude ``kmod``.
    '''
    nvox = np.asarray(nvox, dtype=np.int64)
    boxsize = np.broadcast_to(boxsize, (3,)).astype(np.float64)
    dcell = boxsize / nvox

    kx = 2.0 * np.pi * np.fft.fftfreq(int(nvox[0]), d=dcell[0])
    ky = 2.0 * np.pi * np.fft.fftfreq(int(nvox[1]), d=dcell[1])
    kz_func = np.fft.rfftfreq if hermitian else np.fft.fftfreq
    kz = 2.0 * np.pi * kz_func(int(nvox[2]), d=dcell[2])
    kvec = np.array(np.meshgrid(kx, ky, kz, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod
def _weight_rfft_modes(nvox, shape):
    r'''
    Return conjugate-pair weights for an ``rfftn``-stored Fourier grid.

    Notes
    -----
    For a real-valued field, the full complex Fourier transform obeys
    Hermitian symmetry, so most stored modes in an ``rfftn`` output
    represent both ``+k_z`` and ``-k_z`` and therefore carry weight 2.
    The self-conjugate planes ``k_z = 0`` and, for even ``nvox[2]``, the
    Nyquist plane ``k_z = k_\mathrm{Ny}`` are stored only once and carry
    weight 1.
    '''
    if len(shape) != 3:
        raise ValueError(
            f'Expected a 3D Fourier grid shape, got {shape!r}.'
        )

    nz_stored = nvox[2] // 2 + 1
    if shape[2] != nz_stored:
        raise ValueError(
            'Incompatible rfftn shape for the provided mesh: '
            f'shape[2]={shape[2]}, expected {nz_stored}.'
        )

    weight = 2.0 * np.ones(shape, dtype=np.float64)
    weight[:, :, 0] = 1.0
    if nvox[2] % 2 == 0:
        weight[:, :, nz_stored - 1] = 1.0
    return weight

def _build_k_bin_edges(boxsize, dcell, kmin = None, kmax = None, dk_bin = None):
    r'''
    Construct isotropic shell bin edges for power-spectrum estimation.

    Parameters
    ----------
    boxsize : ndarray of shape (3,)
        Periodic box size.
    dcell : ndarray of shape (3,)
        Cell size in each dimension [Mpc/h].
    kmin, kmax, dk_bin : float or None
        Requested binning configuration. When omitted, the defaults are
        the fundamental mode, the Nyquist mode, and the fundamental-mode
        spacing, respectively.

    Returns
    -------
    ndarray
        Bin edges in wavenumber [h/Mpc].

    Notes
    -----
    The fundamental mode is defined as ``2*pi / max(boxsize)``, i.e. the
    smallest fundamental frequency across all axes.  This is the most
    conservative default for isotropic binning of an anisotropic box. The
    default Nyquist cutoff is the smallest per-axis Nyquist frequency,
    ``min(pi / dcell_i) = pi / max(dcell)``.
    '''
    boxsize = np.broadcast_to(boxsize, (3,)).astype(np.float64)
    dcell = np.broadcast_to(dcell, (3,)).astype(np.float64)

    k_fund = 2.0 * np.pi / np.max(boxsize)
    k_ny = np.pi / np.max(dcell)

    if kmin is None: kmin = k_fund
    if kmax is None: kmax = k_ny
    if dk_bin is None: dk_bin = k_fund

    if kmin <= 0.0:
        raise ValueError(f'kmin must be positive, got {kmin}.')
    if kmax <= kmin:
        raise ValueError(f'kmax must be larger than kmin, got {kmin} >= {kmax}.')
    if dk_bin <= 0.0:
        raise ValueError(f'dk_bin must be positive, got {dk_bin}.')

    bin_edges = np.arange(
        kmin - dk_bin / 2.0, kmax + dk_bin, dk_bin, dtype=np.float64,
    )
    if bin_edges.size < 2:
        raise ValueError('Power-spectrum binning produced fewer than one bin.')
    return bin_edges
def _bin_isotropic_modes(values, kmod, nvox, dcell, boxsize, kmin = None, kmax = None, dk_bin = None):
    r'''
    Bin a per-mode scalar field over isotropic Fourier shells.

    Parameters
    ----------
    values : ndarray
        Scalar quantity defined on the stored ``rfftn`` Fourier grid.
    kmod : ndarray
        Magnitude of the Fourier wavevector at each stored grid point
        [h/Mpc]. Must have the same shape as ``values``.
    nvox : ndarray of shape (3,)
        Number of cells in each dimension of the underlying real-space grid.
    dcell : ndarray of shape (3,)
        Cell size in each dimension [Mpc/h].
    boxsize : ndarray of shape (3,)
        Periodic box size [Mpc/h].
    kmin, kmax, dk_bin : float or None
        Shell-binning configuration.

    Returns
    -------
    k_centres : ndarray
        Weighted mean wavenumber in each populated shell [h/Mpc].
    values_binned : ndarray
        Weighted shell average of ``values``.
    nmodes : ndarray
        Number of independent Fourier modes in each populated shell.
    '''
    if values.shape != kmod.shape:
        raise ValueError(
            'values and kmod must have identical shapes, got '
            f'{values.shape!r} and {kmod.shape!r}.'
        )

    bin_edges = _build_k_bin_edges(
        boxsize, dcell, kmin=kmin, kmax=kmax, dk_bin=dk_bin,
    )
    n_bins = len(bin_edges) - 1

    weight = np.ones(kmod.shape)#_weight_rfft_modes(nvox, kmod.shape)
    k_flat = np.asarray(kmod, dtype=np.float64).ravel()
    values_flat = np.asarray(values, dtype=np.float64).ravel()
    weight_flat = weight.ravel()

    bin_idx = np.digitize(k_flat, bin_edges) - 1
    
    valid = (
        (bin_idx >= 0) & (bin_idx < n_bins)
        & (k_flat > 0.0) & np.isfinite(values_flat)
    )

    idx_valid = bin_idx[valid]
    k_valid = k_flat[valid]
    values_valid = values_flat[valid]
    w_valid = weight_flat[valid]

    nmodes = np.bincount(
        idx_valid, weights=w_valid, minlength=n_bins,
    ).astype(np.int64)

    print(nmodes.shape)
    for i in range(0,len(nmodes)):
        print(nmodes[i])
    
    k_sum = np.bincount(
        idx_valid, weights=w_valid * k_valid, minlength=n_bins,
    ).astype(np.float64)

    print(k_sum.shape)
    for i in range(0,len(k_sum)):
        print(k_sum[i])
    
    values_sum = np.bincount(
        idx_valid, weights=w_valid * values_valid, minlength=n_bins,
    ).astype(np.float64)

    populated = nmodes > 0
    k_centres = np.full(n_bins, np.nan, dtype=np.float64)
    values_binned = np.full(n_bins, np.nan, dtype=np.float64)
    k_centres[populated] = k_sum[populated] / nmodes[populated]
    values_binned[populated] = values_sum[populated] / nmodes[populated]
    return k_centres[populated], values_binned[populated], nmodes[populated]

In [ ]:
kmin=None;kmax=None;dk_bin=None;
nvox=np.array([64,64,64])
boxsize=np.array([50,50,50])
dcell=boxsize/nvox
V_box=np.prod(boxsize)
N_cells=int(np.prod(nvox))
delta_k=readcomp("dft_results3d8C.dat").reshape((64,64,64))
kvec, kmod=_fourier_grid_box(nvox, boxsize)
pk_grid = (V_box / N_cells**2) * np.abs(delta_k)**2
k_centres, pk_binned, nmodes = _bin_isotropic_modes(pk_grid, kmod, nvox, dcell, boxsize, kmin=kmin, kmax=kmax, dk_bin=dk_bin)